In [1]:
import jax.numpy as jnp
from jax import random
from jax import lax

from jax.scipy.special import gammaln, betaln

from numpyro.distributions import constraints
from numpyro.distributions.continuous import Beta, Dirichlet, Gamma
from numpyro.distributions.conjugate import NegativeBinomialProbs
from numpyro.distributions.distribution import Distribution
from numpyro.distributions.util import promote_shapes, validate_sample
from numpyro.util import is_prng_key

In [ ]:
class BetaNegativeBinomial(Distribution):
	r"""
	Beta negative binomial distribution. Also known as inverse Markvo-Polya distribution and the generalized Waring distribution.
 
	The probability mass function is defined as:
	.. math::
		f(n | r, \alpha, \beta) = \frac{\Gamma(n + r)}{n! \Gamma(r)} \frac{\text{B}(\beta + n, \alpha + r)}{\text{B}(\beta, \alpha)}
	
	where :math:`n \in \mathbb{N}` is the count, :math:`r \in \mathbb{R}^+` is the number of success (`total_count`),
	:math:`\alpha \in \mathbb{R}^+` and :math:`\beta \in \mathbb{R}^+` is the concentration parameters of the beta disribtuion (`concentration1` and `concentration0`).,
	:math:`\text{B}` is the beta function, and :math:`\Gamma` is the gamma function.
 	"""
	
	arg_constraints = {
		'total_count': constraints.positive,
		'concentration1': constraints.positive,
		'concentration0': constraints.positive
	}
	support = constraints.positive_integer
	pytree_data_fields = ('total_count', 'concentration1', 'concentration0', '_beta')
 
	def __init__(self, total_count, concentration1, concentration0, *, validate_args=None):
		total_count = jnp.asarray(total_count)
		concentration1 = jnp.asarray(concentration1)
		concentration0 = jnp.asarray(concentration0)
		batch_shape = lax.broadcast_shapes(
			total_count.shape, concentration1.shape, concentration0.shape
		)
		self.total_count, self.concentration1, self.concentration0 = jnp.broadcast_arrays(
			total_count, concentration1, concentration0
		)
		self._beta = Beta(self.concentration1, self.concentration0)
		super(BetaNegativeBinomial, self).__init__(
			batch_shape=batch_shape,
			event_shape=(),
	 		validate_args=validate_args
		)
	
	def sample(self, key, sample_shape=()):
		assert is_prng_key(key)
		key_beta, key_negbin = random.split(key)
		p = self._beta.sample(key_beta, sample_shape)
		q = 1.0 - p
		return NegativeBinomialProbs(self.total_count, q).sample(key_negbin)

	@validate_sample
	def log_prob(self, value):
		return (
			-gammaln(value + 1) + gammaln(value + self.total_count) - gammaln(self.total_count) +
			betaln(value + self.concentration1, self.concentration0 + self.total_count) - betaln(self.concentration1, self.concentration0)
		)

	@property
	def mean(self):
		return jnp.where(
			self.concentration1 > 1,
			self.total_count * self.concentration0 / (self.concentration1 - 1),
			jnp.inf
		)

	@property
	def variance(self):
		return jnp.where(
			self.concentration1 > 2,
			(self.total_count * self.concentration0 * (self.concentration1 + self.concentration0 - 1) /
			((self.concentration1 - 1) ** 2 * (self.concentration1 - 2))),
			jnp.inf
		)

In [68]:
import numpy as np
params = (np.array([1, 2, 3]), np.array([2, 3, 3]), 2.0)
bnb = BetaNegativeBinomial(*params)

In [66]:
bnb.cdf(3)

[3 3 3]


TypeError: Only scalar arrays can be converted to Python scalars; got arr.ndim=1

In [ ]:
np.exp(bnb.log_prob(0)) + np.exp(bnb.log_prob(1)) + np.exp(bnb.log_prob(2)) + np.exp(bnb.log_prob(3) + np.exp(bnb.log_prob(4)))

array([0.85922045, 0.6022904 , 0.4600993 ], dtype=float32)

In [ ]:
from numpyro.util import not_jax_tracer

is_valid = bnb.arg_constraints['concentration1'](0.1)
not_jax_tracer(is_valid)
if not np.all(is_valid):
	raise ValueError('Invalid value')

In [133]:
from numpyro.distributions import GammaPoisson

GammaPoisson(-1, 0.5).sample(random.PRNGKey(0))

Array(-1, dtype=int32)